In [2]:
#### Test para la hipotesis 1 planteada por nosotros#####


import numpy as np
import pandas as pd
from scipy import stats

df = pd.read_parquet("dataset_limpio.parquet")

# 1. Limpieza y preparación de la variable de interés
sodio_clean = df['sodium_100g'].dropna()

# 2. Definición del umbral normativo (0.3g = 300mg)
mu_0 = 0.3

# 3. Análisis Descriptivo Univariado
n_obs = len(sodio_clean)
media = sodio_clean.mean()
mediana = sodio_clean.median()
std = sodio_clean.std()
iqr = stats.iqr(sodio_clean)

print("--- ANÁLISIS DESCRIPTIVO ---")
print(f"N observaciones: {n_obs}")
print(f"Media: {media:.4f} g/100g")
print(f"Mediana: {mediana:.4f} g/100g")
print(f"Desviación Estándar: {std:.4f}")
print(f"Rango Intercuartílico (IQR): {iqr:.4f}")

# 4. Prueba t de Student para una muestra (Unilateral Superior)
t_stat, p_val_t = stats.ttest_1samp(sodio_clean, popmean=mu_0, alternative='greater')

# 5. Complemento No Paramétrico (Prueba de Wilcoxon respecto a la mediana)
res_wilcoxon = stats.wilcoxon(sodio_clean - mu_0, alternative='greater')

print("\n--- RESULTADOS DE LAS PRUEBAS ---")
print(f"Prueba t - Estadístico t: {t_stat:.4f} | p-valor: {p_val_t:.5e}")
print(f"Prueba Wilcoxon - Estadístico: {res_wilcoxon.statistic:.4f} | p-valor: {res_wilcoxon.pvalue:.5e}")

--- ANÁLISIS DESCRIPTIVO ---
N observaciones: 4686
Media: 0.4809 g/100g
Mediana: 0.1200 g/100g
Desviación Estándar: 2.2980
Rango Intercuartílico (IQR): 0.3810

--- RESULTADOS DE LAS PRUEBAS ---
Prueba t - Estadístico t: 5.3883 | p-valor: 3.72979e-08
Prueba Wilcoxon - Estadístico: 3572687.0000 | p-valor: 1.00000e+00


In [ ]:
###test de la hipotesis 2 creada por nosotros###


import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest

df = pd.read_parquet("dataset_limpio.parquet")

# 1. Limpieza y preparación de las variables de interés
df_ecoscore = df[['contains_soy', 'ecoscore_num']].dropna().copy()
df_ecoscore['contains_soy'] = df_ecoscore['contains_soy'].astype(bool)

# 2. Crear la variable binaria de Ecoscore desfavorable (1 o 2 = D/E = 1, 3/4/5 = A/B/C = 0)
df_ecoscore['ecoscore_desfavorable'] = df_ecoscore['ecoscore_num'].isin([1, 2]).astype(int)

# 3. Segmentar los datos en los dos grupos
grupo_con_soja = df_ecoscore[df_ecoscore['contains_soy'] == True]['ecoscore_desfavorable']
grupo_sin_soja = df_ecoscore[df_ecoscore['contains_soy'] == False]['ecoscore_desfavorable']

# 4. Prueba Z de Dos Proporciones (Unilateral: proporción con soja > proporción sin soja)
exitos = [grupo_con_soja.sum(), grupo_sin_soja.sum()]
observaciones = [len(grupo_con_soja), len(grupo_sin_soja)]

z_stat, p_val_prop = proportions_ztest(count=exitos, nobs=observaciones, alternative='larger')

print("--- PRUEBA Z DE DOS PROPORCIONES (Ecoscore Desfavorable 1 y 2) ---")
print(f"Proporción con Soja (Desfavorable): {grupo_con_soja.mean():.4f}")
print(f"Proporción sin Soja (Desfavorable): {grupo_sin_soja.mean():.4f}")
print(f"Estadístico Z: {z_stat:.4f} | p-valor: {p_val_prop:.5e}")

# 5. Prueba Chi-cuadrado de Independencia (para toda la tabla de contingencia 1 a 5 vs Soja)
tabla_contingencia = pd.crosstab(df_ecoscore['contains_soy'], df_ecoscore['ecoscore_num'])
chi2, p_val_chi2, dof, ex = stats.chi2_contingency(tabla_contingencia)

print("\n--- PRUEBA CHI-CUADRADO (Distribución global Ecoscore) ---")
print(f"Estadístico Chi2: {chi2:.4f} | p-valor: {p_val_chi2:.5e}")

# 6. Prueba U de Mann-Whitney (Ordinal: 1=E ... 5=A)
# 'less' evalúa si el grupo con soja tiene valores significativamente menores (peores notas)
grupo_con_soja_num = df_ecoscore[df_ecoscore['contains_soy'] == True]['ecoscore_num']
grupo_sin_soja_num = df_ecoscore[df_ecoscore['contains_soy'] == False]['ecoscore_num']

u_stat, p_val_mann = stats.mannwhitneyu(
    grupo_con_soja_num,
    grupo_sin_soja_num,
    alternative='less'
)

print("\n--- PRUEBA U DE MANN-WHITNEY (Ordinal) ---")
print(f"Estadístico U: {u_stat:.4f} | p-valor: {p_val_mann:.5e}")

In [ ]:
####mapa de calor para la soja y el ecoscore####

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns


df = pd.read_parquet("dataset_limpio.parquet")


# 1. Preparar los datos (filtrar nulos)
df_heatmap_eco = df[['contains_soy', 'ecoscore_num']].dropna().copy()

# Mapeo de etiquetas claras para el gráfico
df_heatmap_eco['Soja'] = df_heatmap_eco['contains_soy'].map(
    {True: 'Con Soja', False: 'Sin Soja'}
)

# Mapeo ordinal: 1=E (peor impacto) a 5=A (mejor impacto)
eco_map = {1: 'E (1)', 2: 'D (2)', 3: 'C (3)', 4: 'B (4)', 5: 'A (5)'}
df_heatmap_eco['Ecoscore'] = df_heatmap_eco['ecoscore_num'].map(eco_map)

# 2. Crear tabla de contingencia en porcentajes (normalizada por filas)
ct_ecoscore = (
    pd.crosstab(
        df_heatmap_eco['Soja'], df_heatmap_eco['Ecoscore'], normalize='index'
    )
    * 100
)

# 3. Graficar el Mapa de Calor
plt.figure(figsize=(9, 5))
sns.heatmap(
    ct_ecoscore,
    annot=True,
    fmt='.1f',
    cmap='YlOrRd',
    cbar_kws={'label': 'Porcentaje (%) dentro del grupo'},
)

plt.title(
    'Distribución Porcentual de Ecoscore según Presencia de Soja',
    fontsize=13,
    pad=15,
)
plt.xlabel('Calificación Ecoscore', fontsize=11)
plt.ylabel('Presencia de Soja', fontsize=11)

plt.tight_layout()
plt.show()